# Example 10 - Cross-Currency Swap Pricing with the ORE Python Addin

This notebook prices a 1-year EUR/USD cross-currency (xccy) basis swap using
the ORE SWIG Python wrapper.  It demonstrates the full ORE Python API:
conventions, curve configurations, programmatic trade construction,
`InputParameters`-based analytics, and in-memory report extraction — all
driven from Python objects with no XML configuration files required.

**Prerequisite:** set `ORESWIG_PKG` to the directory containing `ORE.py`/`_OREP.pyd`.

## 1 - Environment setup

In [ ]:
import sys, os
from pathlib import Path  # imported before "from ORE import *" to avoid shadowing
import pandas as pd

base_dir   = Path.cwd()
OUTPUT_DIR = base_dir / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: set ORESWIG_PKG to a local ORE-SWIG build directory.
# If unset, ORE is expected to be importable from the current Python environment.
# Example (Windows):
#   set ORESWIG_PKG=<build_dir>\\ORE-SWIG;<build_dir>\\ORE-SWIG\\Release
oreswig_pkg = os.environ.get("ORESWIG_PKG", "")
if oreswig_pkg:
    sys.path += oreswig_pkg.split(";")

from ORE import *  # "Path" is now shadowed by ORE.Path; use base_dir (already set)
import ORE

print("ORE loaded from:", ORE.__file__)

## 2 - Build market configurations in Python

We define conventions, pricing engines, curve configurations and today's market
parameters entirely in Python using the ORE SWIG API — no XML files required.

In [ ]:
# ── Conventions ──────────────────────────────────────────────────────────
# Only the zero-rate conventions referenced by the curve configs are needed,
# plus the FX convention for the xccy swap's FXReset index.
convs = Conventions()
convs.add(ZeroRateConvention(
    "EUR-ZERO-TENOR-BASED", "A365", "TARGET", "Continuous", "Daily",
    "2", "TARGET", "Following"
))
convs.add(ZeroRateConvention(
    "USD-ZERO-TENOR-BASED", "A365", "US", "Continuous", "Daily",
    "2", "US", "Following"
))
convs.add(FXConvention("FX-ECB-USD-EUR", "2", "USD", "EUR", "10000", "US,TARGET"))

# ── Pricing Engine ─────────────────────────────────────────────────────
engine_data = EngineData()
for prod, model, eng in [
    ("Swap",              "DiscountedCashflows", "DiscountingSwapEngine"),
    ("CrossCurrencySwap", "DiscountedCashflows", "DiscountingCrossCurrencySwapEngine"),
]:
    engine_data.setModel(prod, model)
    engine_data.setEngine(prod, eng)
    engine_data.setModelParameters(prod, StringStringMap({}))
    engine_data.setEngineParameters(prod, StringStringMap({}))

# ── Curve Configurations ───────────────────────────────────────────────
# Three yield curves, each bootstrapped from direct zero-rate quotes.
# Wildcard quotes (trailing *) match all dates in market data.
curve_configs = CurveConfigurations()

for curve_id, ccy, conv_id, quote_prefix in [
    ("USD-SOFR",   "USD", "USD-ZERO-TENOR-BASED", "ZERO/RATE/USD/USD-SOFR/A365/*"),
    ("EUR-ESTER",  "EUR", "EUR-ZERO-TENOR-BASED", "ZERO/RATE/EUR/EUR-ESTER/A365/*"),
    ("EUR-IN-USD", "EUR", "EUR-ZERO-TENOR-BASED", "ZERO/RATE/EUR/EUR-IN-USD/A365/*"),
]:
    seg  = DirectYieldCurveSegment("Zero", conv_id, StrVector([quote_prefix]))
    segs = YieldCurveSegmentVector()
    segs.append(seg)
    ycc  = YieldCurveConfig(curve_id, f"{ccy} {curve_id} curve", ccy, "",
                            segs, "Discount", "NaturalCubic", "A365", True)
    curve_configs.add(CurveSpec.CurveType_Yield, curve_id, ycc)

# ── Today's Market Parameters ──────────────────────────────────────────
# Defines how market objects (discount curves, index curves, FX spots)
# are resolved under the "default" and "xois" configurations.
tm = TodaysMarketParameters()

# "default" configuration: EUR discounted at EUR-ESTER
default_cfg = MarketConfiguration()
tm.addConfiguration("default", default_cfg)
tm.addMarketObject(MarketObject_DiscountCurve, "default",
    StringStringMap({"EUR": "Yield/EUR/EUR-ESTER", "USD": "Yield/USD/USD-SOFR"}))
tm.addMarketObject(MarketObject_IndexCurve, "default",
    StringStringMap({"EUR-ESTER": "Yield/EUR/EUR-ESTER",
                     "USD-SOFR":  "Yield/USD/USD-SOFR"}))
tm.addMarketObject(MarketObject_FXSpot, "default",
    StringStringMap({"EURUSD": "FX/EUR/USD"}))

# "xois" configuration: EUR discounted at EUR-IN-USD (cross-currency implied)
xois_cfg = MarketConfiguration()
xois_cfg.setId(MarketObject_DiscountCurve, "xois")
tm.addConfiguration("xois", xois_cfg)
tm.addMarketObject(MarketObject_DiscountCurve, "xois",
    StringStringMap({"EUR": "Yield/EUR/EUR-IN-USD", "USD": "Yield/USD/USD-SOFR"}))

print("All configurations built in Python:")
for key in ["conventions", "pricing_engine", "curve_config", "todays_market"]:
    print(f"  {key}: OK")

## 3 - Inspect conventions using the ORE Python API

The `Conventions` object exposes individual entries by id.
We verify that the three conventions needed for pricing are registered.

In [ ]:
# Verify conventions
ids_to_check = ["EUR-ZERO-TENOR-BASED", "USD-ZERO-TENOR-BASED", "FX-ECB-USD-EUR"]
for cid in ids_to_check:
    if convs.has(cid):
        print(f"  {cid} : OK")
    else:
        print(f"  {cid} : *** NOT FOUND ***")

## 4 - Inspect yield curve configurations

Retrieve the `CurveConfigurations` object and query the yield curves
required to price EUR/USD xccy swaps.

In [ ]:
for name in ["USD-SOFR", "EUR-ESTER", "EUR-IN-USD"]:
    try:
        ycc = curve_configs.yieldCurveConfig(name)
        print(name, "| currency:", ycc.currency(),
              "| interpolation:", ycc.interpolationMethod())
    except Exception as e:
        print(name, "error:", e)

## 5 - Build the trade in Python

We build a 1-year EUR/USD cross-currency swap entirely in Python using
`ScheduleData`, `FloatingLegData`, `LegData`, `Envelope`, and `ORESwap`.

This trade object is passed directly to `InputParameters` in sections 7 and 9.

In [ ]:
# Build a 1-year EUR/USD cross-currency swap in Python.
ASOF  = "2026-03-03"
START = "2025-11-24"
END   = "2026-11-24"

# Schedule: 3M tenor, backward generation, modified following
schedule = ScheduleData(
    ScheduleRules(START, END, "3M", "NYB,TGT", "MF", "MF", "Backward")
)

# EUR floating leg: payer, EUR-ESTER, in arrears, -2.25 bp spread, rateCutoff=0
eur_float = FloatingLegData(
    "EUR-ESTER", 0, True,
    DoubleVector([-0.000225]),  # spreads
    StrVector(),                # spreadDates
    DoubleVector(),             # caps
    StrVector(),                # capDates
    DoubleVector(),             # floors
    StrVector(),                # floorDates
    DoubleVector([1.0]),        # gearings
    StrVector(),                # gearingDates
    False,                      # isAveraged
    False,                      # nakedOption
    False,                      # hasSubPeriods
    False,                      # includeSpread
    Period(0, Days),            # lookback
    0                           # rateCutoff
)
eur_leg = LegData(
    eur_float, True, "EUR", schedule, "A360",
    DoubleVector([200000000.0]), StrVector(), "MF",
    True, True, False,          # exchanges: initial, final, amortizing
    True,                       # isNotResetXCCY (EUR leg is not FX-reset)
    "", 0.0, "",                # foreignCurrency, foreignAmount, resetStartDate
    "",                         # fxIndex
    AmortizationDataVector(),   # amortizationData
    "2"                         # paymentLag
)

# USD floating leg: receiver, USD-SOFR, in arrears, 0 spread, FX reset, rateCutoff=0
usd_float = FloatingLegData(
    "USD-SOFR", 0, True,
    DoubleVector([0.0]),        # spreads
    StrVector(),                # spreadDates
    DoubleVector(),             # caps
    StrVector(),                # capDates
    DoubleVector(),             # floors
    StrVector(),                # floorDates
    DoubleVector([1.0]),        # gearings
    StrVector(),                # gearingDates
    False,                      # isAveraged
    False,                      # nakedOption
    False,                      # hasSubPeriods
    False,                      # includeSpread
    Period(0, Days),            # lookback
    0                           # rateCutoff
)
usd_leg = LegData(
    usd_float, False, "USD", schedule, "A360",
    DoubleVector([230800000.0]), StrVector(), "MF",
    True, True, False,          # exchanges: initial, final, amortizing
    False,                      # isNotResetXCCY=False -> FX-resettable leg
    "EUR", 200000000.0, "",     # foreignCurrency, foreignAmount, resetStartDate
    "FX-ECB-USD-EUR",           # fxIndex
    AmortizationDataVector(),   # amortizationData
    "2"                         # paymentLag
)

# Envelope matching the XML trade
additional = StringStringMap({
    "valuation_date": "2026-03-03",
    "trade_type": "GENERICCROSSCURRENCYSWAP",
})
env = Envelope("3000558_RV", "3000558_RV", additional)

xccy_swap = ORESwap(env, eur_leg, usd_leg)
xccy_swap.setId("268128NY")
# print(xccy_swap.toXMLString())

## 6 - Market data & fixings

Minimal market data defined inline: FX spot, daily zero-rate curves for
USD-SOFR, EUR-ESTER, and EUR-IN-USD, plus daily OIS/FX fixings from trade
start to ASOF.

In [ ]:
# Minimal market data: daily quotes needed for this 1Y EUR/USD xccy swap (ASOF 2026-03-03)
# FX spot + daily zero-rate curves for USD-SOFR, EUR-ESTER, EUR-IN-USD (2026)
MARKET_DATA = """
20260303 FX/RATE/EUR/USD 1.1612
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-04 0.037613
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-05 0.037396
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-06 0.037324
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-09 0.037249
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-10 0.037239
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-11 0.037232
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-12 0.037226
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-13 0.037221
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-16 0.03721
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-17 0.037208
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-18 0.037206
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-19 0.037204
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-20 0.037203
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-23 0.037199
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-24 0.037198
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-25 0.037197
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-26 0.037196
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-27 0.037195
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-30 0.037193
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-03-31 0.037193
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-01 0.037194
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-02 0.037191
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-03 0.037188
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-06 0.037181
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-07 0.037178
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-08 0.037176
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-09 0.037175
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-10 0.037173
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-13 0.037168
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-14 0.037166
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-15 0.037165
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-16 0.037163
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-17 0.037162
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-20 0.037158
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-21 0.037157
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-22 0.037156
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-23 0.037155
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-24 0.037154
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-27 0.037152
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-28 0.037151
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-29 0.03715
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-04-30 0.037149
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-01 0.037149
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-04 0.037132
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-05 0.037127
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-06 0.037122
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-07 0.037117
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-08 0.037112
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-11 0.037099
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-12 0.037094
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-13 0.03709
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-14 0.037086
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-15 0.037082
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-18 0.037071
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-19 0.037068
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-20 0.037064
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-21 0.037061
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-22 0.037058
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-26 0.037045
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-27 0.037043
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-28 0.03704
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-05-29 0.037037
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-01 0.03703
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-02 0.037024
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-03 0.037018
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-04 0.037012
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-05 0.037007
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-08 0.036991
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-09 0.036986
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-10 0.036981
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-11 0.036976
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-12 0.036971
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-15 0.036957
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-16 0.036953
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-17 0.036949
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-18 0.036944
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-22 0.036928
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-23 0.036924
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-24 0.03692
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-25 0.036916
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-26 0.036913
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-29 0.036902
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-06-30 0.036899
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-01 0.036896
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-02 0.036888
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-03 0.03688
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-06 0.036857
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-07 0.03685
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-08 0.036842
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-09 0.036835
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-10 0.036828
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-13 0.036808
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-14 0.036801
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-15 0.036795
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-16 0.036788
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-17 0.036782
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-20 0.036763
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-21 0.036757
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-22 0.036752
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-23 0.036746
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-24 0.03674
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-27 0.036723
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-28 0.036718
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-29 0.036713
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-30 0.036707
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-07-31 0.036702
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-03 0.036687
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-04 0.036678
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-05 0.036668
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-06 0.036658
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-07 0.036649
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-10 0.036621
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-11 0.036612
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-12 0.036603
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-13 0.036594
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-14 0.036585
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-17 0.03656
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-18 0.036551
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-19 0.036543
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-20 0.036535
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-21 0.036527
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-24 0.036504
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-25 0.036496
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-26 0.036488
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-27 0.036481
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-28 0.036473
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-08-31 0.036452
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-01 0.036445
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-02 0.036435
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-03 0.036426
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-04 0.036417
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-08 0.03638
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-09 0.036371
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-10 0.036363
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-11 0.036354
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-14 0.036329
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-15 0.03632
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-16 0.036312
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-17 0.036304
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-18 0.036296
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-21 0.036272
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-22 0.036265
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-23 0.036257
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-24 0.036249
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-25 0.036242
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-28 0.03622
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-29 0.036212
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-09-30 0.036205
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-01 0.036198
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-02 0.036188
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-05 0.036159
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-06 0.036149
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-07 0.036139
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-08 0.03613
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-09 0.03612
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-13 0.036083
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-14 0.036074
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-15 0.036065
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-16 0.036056
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-19 0.03603
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-20 0.036021
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-21 0.036013
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-22 0.036004
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-23 0.035996
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-26 0.035971
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-27 0.035963
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-28 0.035955
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-29 0.035947
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-10-30 0.035939
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-02 0.035916
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-03 0.035906
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-04 0.035896
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-05 0.035886
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-06 0.035876
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-09 0.035847
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-10 0.035837
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-12 0.035818
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-13 0.035809
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-16 0.035781
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-17 0.035772
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-18 0.035763
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-19 0.035754
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-20 0.035746
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-23 0.035719
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-24 0.035711
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-25 0.035702
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-27 0.035685
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-11-30 0.035661
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-01 0.035653
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-02 0.035644
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-03 0.035634
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-04 0.035625
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-07 0.035598
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-08 0.035588
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-09 0.035579
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-10 0.03557
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-11 0.035561
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-14 0.035533
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-15 0.035524
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-16 0.035514
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-17 0.035505
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-18 0.035496
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-21 0.035469
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-22 0.03546
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-23 0.035451
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-24 0.035442
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-28 0.035406
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-29 0.035397
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-30 0.035388
20260303 ZERO/RATE/USD/USD-SOFR/A365/2026-12-31 0.03538
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-04 0.019608
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-05 0.019608
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-06 0.019599
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-09 0.019576
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-10 0.019572
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-11 0.01957
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-12 0.019569
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-13 0.019571
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-16 0.019583
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-17 0.019587
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-18 0.019591
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-19 0.019593
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-20 0.019595
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-23 0.019597
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-24 0.019598
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-25 0.019598
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-26 0.019599
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-27 0.0196
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-30 0.019604
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-03-31 0.019605
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-01 0.019606
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-02 0.019607
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-07 0.019611
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-08 0.019612
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-09 0.019613
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-10 0.019614
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-13 0.019616
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-14 0.019617
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-15 0.019618
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-16 0.019618
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-17 0.019619
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-20 0.01962
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-21 0.019621
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-22 0.019621
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-23 0.019622
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-24 0.019622
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-27 0.019624
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-28 0.019624
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-29 0.019624
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-04-30 0.019625
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-04 0.019626
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-05 0.019626
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-06 0.019629
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-07 0.019631
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-08 0.019634
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-11 0.01964
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-12 0.019643
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-13 0.019645
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-14 0.019647
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-15 0.019649
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-18 0.019654
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-19 0.019656
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-20 0.019658
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-21 0.019659
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-22 0.019661
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-25 0.019665
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-26 0.019667
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-27 0.019668
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-28 0.01967
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-05-29 0.019671
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-01 0.019675
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-02 0.019676
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-03 0.019677
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-04 0.019679
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-05 0.01968
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-08 0.019686
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-09 0.019689
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-10 0.019691
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-11 0.019693
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-12 0.019695
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-15 0.019701
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-16 0.019702
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-17 0.019704
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-18 0.019706
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-19 0.019708
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-22 0.019713
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-23 0.019714
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-24 0.019716
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-25 0.019718
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-26 0.019719
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-29 0.019724
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-06-30 0.019725
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-01 0.019727
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-02 0.019728
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-03 0.019729
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-06 0.019733
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-07 0.019736
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-08 0.019738
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-09 0.01974
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-10 0.019742
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-13 0.019748
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-14 0.01975
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-15 0.019752
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-16 0.019754
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-17 0.019756
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-20 0.019761
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-21 0.019763
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-22 0.019765
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-23 0.019767
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-24 0.019768
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-27 0.019773
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-28 0.019775
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-29 0.019776
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-30 0.019778
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-07-31 0.01978
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-03 0.019784
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-04 0.019786
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-05 0.019787
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-06 0.019789
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-07 0.019791
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-10 0.019796
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-11 0.019798
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-12 0.0198
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-13 0.019802
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-14 0.019804
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-17 0.019809
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-18 0.01981
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-19 0.019812
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-20 0.019814
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-21 0.019815
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-24 0.01982
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-25 0.019822
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-26 0.019823
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-27 0.019825
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-28 0.019826
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-08-31 0.01983
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-01 0.019832
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-02 0.019833
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-03 0.019835
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-04 0.019836
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-07 0.01984
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-08 0.019842
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-09 0.019844
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-10 0.019846
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-11 0.019847
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-14 0.019853
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-15 0.019854
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-16 0.019856
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-17 0.019858
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-18 0.019859
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-21 0.019864
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-22 0.019866
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-23 0.019867
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-24 0.019869
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-25 0.01987
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-28 0.019875
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-29 0.019876
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-09-30 0.019878
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-01 0.019879
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-02 0.019881
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-05 0.019885
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-06 0.019886
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-07 0.019888
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-08 0.019889
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-09 0.019891
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-12 0.019895
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-13 0.019897
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-14 0.019898
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-15 0.0199
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-16 0.019901
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-19 0.019905
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-20 0.019907
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-21 0.019908
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-22 0.019909
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-23 0.019911
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-26 0.019915
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-27 0.019916
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-28 0.019917
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-29 0.019919
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-10-30 0.01992
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-02 0.019924
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-03 0.019926
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-04 0.019927
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-05 0.019928
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-06 0.01993
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-09 0.019934
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-10 0.019935
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-11 0.019936
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-12 0.019938
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-13 0.019939
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-16 0.019943
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-17 0.019944
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-18 0.019946
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-19 0.019947
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-20 0.019948
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-23 0.019952
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-24 0.019954
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-25 0.019955
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-26 0.019956
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-27 0.019957
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-11-30 0.019961
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-01 0.019963
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-02 0.019964
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-03 0.019965
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-04 0.019967
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-07 0.019971
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-08 0.019972
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-09 0.019974
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-10 0.019975
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-11 0.019976
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-14 0.01998
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-15 0.019982
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-16 0.019983
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-17 0.019984
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-18 0.019986
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-21 0.01999
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-22 0.019991
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-23 0.019992
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-24 0.019993
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-28 0.019998
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-29 0.019999
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-30 0.020001
20260303 ZERO/RATE/EUR/EUR-ESTER/A365/2026-12-31 0.020002
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-04 0.019583
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-05 0.022228
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-06 0.021203
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-09 0.020159
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-10 0.020008
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-11 0.019894
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-12 0.019806
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-13 0.019735
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-16 0.019585
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-17 0.01955
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-18 0.019521
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-19 0.019496
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-20 0.019475
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-23 0.019429
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-24 0.019418
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-25 0.019407
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-26 0.019397
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-27 0.019431
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-30 0.019518
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-03-31 0.019356
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-01 0.019046
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-02 0.018913
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-07 0.019238
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-08 0.019252
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-09 0.019263
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-10 0.019272
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-13 0.019288
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-14 0.01929
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-15 0.019291
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-16 0.01929
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-17 0.019289
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-20 0.019279
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-21 0.019275
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-22 0.019269
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-23 0.019263
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-24 0.019257
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-27 0.019235
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-28 0.019227
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-29 0.019219
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-04-30 0.01921
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-04 0.019161
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-05 0.019182
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-06 0.019195
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-07 0.019207
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-08 0.019217
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-11 0.019237
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-12 0.019242
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-13 0.019245
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-14 0.019247
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-15 0.019248
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-18 0.019248
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-19 0.019246
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-20 0.019244
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-21 0.019241
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-22 0.019238
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-25 0.019226
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-26 0.019221
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-27 0.019216
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-28 0.019211
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-05-29 0.019206
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-01 0.019182
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-02 0.01918
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-03 0.019182
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-04 0.019185
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-05 0.019188
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-08 0.019195
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-09 0.019199
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-10 0.019203
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-11 0.019208
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-12 0.019212
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-15 0.019226
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-16 0.01923
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-17 0.019234
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-18 0.019237
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-19 0.01924
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-22 0.019246
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-23 0.019246
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-24 0.019246
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-25 0.019245
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-26 0.019242
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-29 0.01923
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-06-30 0.019223
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-01 0.019213
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-02 0.019207
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-03 0.019206
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-06 0.019207
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-07 0.019207
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-08 0.019208
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-09 0.019209
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-10 0.01921
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-13 0.019214
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-14 0.019216
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-15 0.019218
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-16 0.01922
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-17 0.019222
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-20 0.019228
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-21 0.01923
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-22 0.019231
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-23 0.019232
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-24 0.019233
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-27 0.019235
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-28 0.019235
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-29 0.019234
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-30 0.019233
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-07-31 0.019232
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-03 0.019224
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-04 0.01923
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-05 0.019238
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-06 0.019242
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-07 0.019246
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-10 0.019257
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-11 0.019259
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-12 0.019262
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-13 0.019264
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-14 0.019265
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-17 0.019269
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-18 0.019269
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-19 0.01927
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-20 0.01927
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-21 0.01927
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-24 0.01927
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-25 0.01927
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-26 0.019269
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-27 0.019268
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-28 0.019268
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-08-31 0.019266
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-01 0.019264
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-02 0.019263
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-03 0.019261
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-04 0.01926
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-07 0.019258
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-08 0.019257
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-09 0.019258
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-10 0.019258
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-11 0.01926
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-14 0.019264
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-15 0.019267
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-16 0.019269
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-17 0.019271
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-18 0.019273
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-21 0.01928
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-22 0.019281
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-23 0.019283
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-24 0.019285
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-25 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-28 0.019287
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-29 0.019287
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-09-30 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-01 0.019283
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-02 0.019283
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-05 0.019287
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-06 0.019287
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-07 0.019287
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-08 0.019287
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-09 0.019288
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-12 0.019288
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-13 0.019287
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-14 0.019287
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-15 0.019287
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-16 0.019287
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-19 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-20 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-21 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-22 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-23 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-26 0.019285
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-27 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-28 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-29 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-10-30 0.019286
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-02 0.019288
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-03 0.019291
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-04 0.019296
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-05 0.019299
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-06 0.019301
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-09 0.019301
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-10 0.019299
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-11 0.019297
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-12 0.019295
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-13 0.019292
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-16 0.019284
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-17 0.019281
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-18 0.019279
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-19 0.019277
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-20 0.019275
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-23 0.019274
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-24 0.019275
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-25 0.019277
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-26 0.019281
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-27 0.019285
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-11-30 0.019306
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-01 0.019321
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-02 0.019326
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-03 0.019327
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-04 0.019325
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-07 0.019317
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-08 0.019318
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-09 0.019321
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-10 0.019326
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-11 0.019333
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-14 0.019359
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-15 0.019369
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-16 0.019379
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-17 0.019388
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-18 0.019397
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-21 0.019417
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-22 0.01942
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-23 0.019421
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-24 0.01942
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-28 0.019387
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-29 0.01937
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-30 0.019349
20260303 ZERO/RATE/EUR/EUR-IN-USD/A365/2026-12-31 0.019324
""".strip().split("\n")

# Fixings: daily EUR-ESTER, USD-SOFR, and FX-ECB-EUR-USD from trade start to ASOF
FIXINGS = """
2025-11-24 EUR-ESTER 0.01924
2025-11-25 EUR-ESTER 0.01926
2025-11-26 EUR-ESTER 0.01928
2025-11-27 EUR-ESTER 0.01927
2025-11-28 EUR-ESTER 0.01925
2025-12-01 EUR-ESTER 0.0193
2025-12-02 EUR-ESTER 0.01929
2025-12-03 EUR-ESTER 0.01926
2025-12-04 EUR-ESTER 0.01929
2025-12-05 EUR-ESTER 0.01929
2025-12-08 EUR-ESTER 0.0193
2025-12-09 EUR-ESTER 0.01929
2025-12-10 EUR-ESTER 0.01929
2025-12-11 EUR-ESTER 0.0193
2025-12-12 EUR-ESTER 0.01932
2025-12-15 EUR-ESTER 0.0193
2025-12-16 EUR-ESTER 0.01932
2025-12-17 EUR-ESTER 0.01932
2025-12-18 EUR-ESTER 0.01931
2025-12-19 EUR-ESTER 0.01933
2025-12-22 EUR-ESTER 0.01932
2025-12-23 EUR-ESTER 0.01925
2025-12-24 EUR-ESTER 0.01926
2025-12-25 EUR-ESTER 0.01926
2025-12-26 EUR-ESTER 0.01926
2025-12-29 EUR-ESTER 0.01934
2025-12-30 EUR-ESTER 0.0193
2025-12-31 EUR-ESTER 0.01921
2026-01-01 EUR-ESTER 0.01921
2026-01-02 EUR-ESTER 0.01936
2026-01-05 EUR-ESTER 0.01933
2026-01-06 EUR-ESTER 0.01933
2026-01-07 EUR-ESTER 0.01933
2026-01-08 EUR-ESTER 0.01933
2026-01-09 EUR-ESTER 0.01932
2026-01-12 EUR-ESTER 0.01931
2026-01-13 EUR-ESTER 0.0193
2026-01-14 EUR-ESTER 0.01931
2026-01-15 EUR-ESTER 0.0193
2026-01-16 EUR-ESTER 0.0193
2026-01-19 EUR-ESTER 0.01929
2026-01-20 EUR-ESTER 0.01932
2026-01-21 EUR-ESTER 0.01932
2026-01-22 EUR-ESTER 0.01933
2026-01-23 EUR-ESTER 0.01933
2026-01-26 EUR-ESTER 0.01934
2026-01-27 EUR-ESTER 0.01934
2026-01-28 EUR-ESTER 0.01933
2026-01-29 EUR-ESTER 0.01933
2026-01-30 EUR-ESTER 0.01926
2026-02-02 EUR-ESTER 0.01933
2026-02-03 EUR-ESTER 0.01931
2026-02-04 EUR-ESTER 0.01932
2026-02-05 EUR-ESTER 0.01931
2026-02-06 EUR-ESTER 0.0193
2026-02-09 EUR-ESTER 0.0193
2026-02-10 EUR-ESTER 0.01929
2026-02-11 EUR-ESTER 0.0193
2026-02-12 EUR-ESTER 0.01931
2026-02-13 EUR-ESTER 0.0193
2026-02-16 EUR-ESTER 0.01929
2026-02-17 EUR-ESTER 0.01931
2026-02-18 EUR-ESTER 0.01931
2026-02-19 EUR-ESTER 0.01933
2026-02-20 EUR-ESTER 0.01932
2026-02-23 EUR-ESTER 0.01931
2026-02-24 EUR-ESTER 0.01932
2026-02-25 EUR-ESTER 0.01933
2026-02-26 EUR-ESTER 0.01935
2026-02-27 EUR-ESTER 0.0193
2026-03-02 EUR-ESTER 0.01934
2026-03-03 EUR-ESTER 0.01934
2025-11-24 USD-SOFR 0.0396
2025-11-25 USD-SOFR 0.0401
2025-11-26 USD-SOFR 0.0405
2025-11-27 USD-SOFR 0.0405
2025-11-28 USD-SOFR 0.0412
2025-12-01 USD-SOFR 0.0412
2025-12-02 USD-SOFR 0.0401
2025-12-03 USD-SOFR 0.0395
2025-12-04 USD-SOFR 0.0392
2025-12-05 USD-SOFR 0.0393
2025-12-08 USD-SOFR 0.0395
2025-12-09 USD-SOFR 0.0393
2025-12-10 USD-SOFR 0.039
2025-12-11 USD-SOFR 0.0366
2025-12-12 USD-SOFR 0.0367
2025-12-15 USD-SOFR 0.0375
2025-12-16 USD-SOFR 0.0369
2025-12-17 USD-SOFR 0.0369
2025-12-18 USD-SOFR 0.0366
2025-12-19 USD-SOFR 0.0366
2025-12-22 USD-SOFR 0.0368
2025-12-23 USD-SOFR 0.0366
2025-12-24 USD-SOFR 0.0366
2025-12-25 USD-SOFR 0.0366
2025-12-26 USD-SOFR 0.0376
2025-12-29 USD-SOFR 0.0377
2025-12-30 USD-SOFR 0.0371
2025-12-31 USD-SOFR 0.0387
2026-01-01 USD-SOFR 0.0387
2026-01-02 USD-SOFR 0.0375
2026-01-05 USD-SOFR 0.037
2026-01-06 USD-SOFR 0.0366
2026-01-07 USD-SOFR 0.0365
2026-01-08 USD-SOFR 0.0364
2026-01-09 USD-SOFR 0.0364
2026-01-12 USD-SOFR 0.0364
2026-01-13 USD-SOFR 0.0365
2026-01-14 USD-SOFR 0.0364
2026-01-15 USD-SOFR 0.0366
2026-01-16 USD-SOFR 0.0365
2026-01-19 USD-SOFR 0.0365
2026-01-20 USD-SOFR 0.0364
2026-01-21 USD-SOFR 0.0363
2026-01-22 USD-SOFR 0.0364
2026-01-23 USD-SOFR 0.0365
2026-01-26 USD-SOFR 0.0366
2026-01-27 USD-SOFR 0.0366
2026-01-28 USD-SOFR 0.0364
2026-01-29 USD-SOFR 0.0365
2026-01-30 USD-SOFR 0.0368
2026-02-02 USD-SOFR 0.0369
2026-02-03 USD-SOFR 0.0369
2026-02-04 USD-SOFR 0.0365
2026-02-05 USD-SOFR 0.0365
2026-02-06 USD-SOFR 0.0364
2026-02-09 USD-SOFR 0.0363
2026-02-10 USD-SOFR 0.0365
2026-02-11 USD-SOFR 0.0365
2026-02-12 USD-SOFR 0.0365
2026-02-13 USD-SOFR 0.0366
2026-02-16 USD-SOFR 0.0366
2026-02-17 USD-SOFR 0.0371
2026-02-18 USD-SOFR 0.0373
2026-02-19 USD-SOFR 0.0367
2026-02-20 USD-SOFR 0.0366
2026-02-23 USD-SOFR 0.0366
2026-02-24 USD-SOFR 0.0367
2026-02-25 USD-SOFR 0.0367
2026-02-26 USD-SOFR 0.0367
2026-02-27 USD-SOFR 0.0368
2026-03-02 USD-SOFR 0.0371
2026-03-03 USD-SOFR 0.0371
2025-11-24 FX-ECB-EUR-USD 1.15405
2025-11-25 FX-ECB-EUR-USD 1.15375
2025-11-26 FX-ECB-EUR-USD 1.1573
2025-11-27 FX-ECB-EUR-USD 1.15845
2025-11-28 FX-ECB-EUR-USD 1.156
2025-12-01 FX-ECB-EUR-USD 1.1628
2025-12-02 FX-ECB-EUR-USD 1.1606
2025-12-03 FX-ECB-EUR-USD 1.16605
2025-12-04 FX-ECB-EUR-USD 1.16665
2025-12-05 FX-ECB-EUR-USD 1.16505
2025-12-08 FX-ECB-EUR-USD 1.1651
2025-12-09 FX-ECB-EUR-USD 1.16445
2025-12-10 FX-ECB-EUR-USD 1.1635
2025-12-11 FX-ECB-EUR-USD 1.17075
2025-12-12 FX-ECB-EUR-USD 1.1727
2025-12-15 FX-ECB-EUR-USD 1.17425
2025-12-16 FX-ECB-EUR-USD 1.17575
2025-12-17 FX-ECB-EUR-USD 1.17255
2025-12-18 FX-ECB-EUR-USD 1.17245
2025-12-19 FX-ECB-EUR-USD 1.17165
2025-12-22 FX-ECB-EUR-USD 1.1731
2025-12-23 FX-ECB-EUR-USD 1.1799
2025-12-24 FX-ECB-EUR-USD 1.17745
2025-12-25 FX-ECB-EUR-USD 1.17745
2025-12-26 FX-ECB-EUR-USD 1.17745
2025-12-29 FX-ECB-EUR-USD 1.1765
2025-12-30 FX-ECB-EUR-USD 1.1774
2025-12-31 FX-ECB-EUR-USD 1.1752
2026-01-01 FX-ECB-EUR-USD 1.1752
2026-01-02 FX-ECB-EUR-USD 1.17215
2026-01-05 FX-ECB-EUR-USD 1.16885
2026-01-06 FX-ECB-EUR-USD 1.17125
2026-01-07 FX-ECB-EUR-USD 1.1688
2026-01-08 FX-ECB-EUR-USD 1.168
2026-01-09 FX-ECB-EUR-USD 1.16465
2026-01-12 FX-ECB-EUR-USD 1.16855
2026-01-13 FX-ECB-EUR-USD 1.16655
2026-01-14 FX-ECB-EUR-USD 1.16555
2026-01-15 FX-ECB-EUR-USD 1.16365
2026-01-16 FX-ECB-EUR-USD 1.16135
2026-01-19 FX-ECB-EUR-USD 1.1625
2026-01-20 FX-ECB-EUR-USD 1.17305
2026-01-21 FX-ECB-EUR-USD 1.1707
2026-01-22 FX-ECB-EUR-USD 1.16945
2026-01-23 FX-ECB-EUR-USD 1.1734
2026-01-26 FX-ECB-EUR-USD 1.186
2026-01-27 FX-ECB-EUR-USD 1.18895
2026-01-28 FX-ECB-EUR-USD 1.19885
2026-01-29 FX-ECB-EUR-USD 1.19415
2026-01-30 FX-ECB-EUR-USD 1.19205
2026-02-02 FX-ECB-EUR-USD 1.18665
2026-02-03 FX-ECB-EUR-USD 1.17875
2026-02-04 FX-ECB-EUR-USD 1.1813
2026-02-05 FX-ECB-EUR-USD 1.17875
2026-02-06 FX-ECB-EUR-USD 1.17925
2026-02-09 FX-ECB-EUR-USD 1.18585
2026-02-10 FX-ECB-EUR-USD 1.19115
2026-02-11 FX-ECB-EUR-USD 1.1919
2026-02-12 FX-ECB-EUR-USD 1.1877
2026-02-13 FX-ECB-EUR-USD 1.18565
2026-02-16 FX-ECB-EUR-USD 1.18665
2026-02-17 FX-ECB-EUR-USD 1.18435
2026-02-18 FX-ECB-EUR-USD 1.18315
2026-02-19 FX-ECB-EUR-USD 1.17885
2026-02-20 FX-ECB-EUR-USD 1.1763
2026-02-23 FX-ECB-EUR-USD 1.1798
2026-02-24 FX-ECB-EUR-USD 1.17855
2026-02-25 FX-ECB-EUR-USD 1.17795
2026-02-26 FX-ECB-EUR-USD 1.1798
2026-02-27 FX-ECB-EUR-USD 1.1799
2026-03-02 FX-ECB-EUR-USD 1.17375
2026-03-03 FX-ECB-EUR-USD 1.15925
""".strip().split("\n")

print(f"Market data: {len(MARKET_DATA)} quotes")
print(f"Fixings:     {len(FIXINGS)} entries")

## 7 - Run NPV analytics

We assemble an `InputParameters` object from the Python-built objects in
sections 2 and 5 and run `OREApp`.

`setMarketConfigs` with `"pricing": "xois"` tells ORE to use the `xois`
market configuration for pricing — discounting the EUR leg with the
cross-currency implied curve `EUR-IN-USD` rather than `EUR-ESTER`.
This is the standard CSA discounting convention for EUR/USD xccy swaps.

In [ ]:
# Build a portfolio from the Python trade constructed in section 5.
portfolio = Portfolio()
portfolio.add(xccy_swap)

# Market configuration map: controls which TodaysMarket configuration is used
# for each analytic.  "pricing": "xois" selects the xois configuration so that
# EUR cash flows are discounted at EUR-IN-USD (the cross-currency implied curve)
# rather than EUR-ESTER (the standard CSA discounting convention).
market_configs = StringStringMap({
    "lgmcalibration": "inccy",
    "fxcalibration":  "default",
    "eqcalibration":  "default",
    "pricing":        "xois",
    "simulation":     "xois",
    "sensitivity":    "xois",
})

# Wire up InputParameters from Python objects in sections 2 and 5.
ip_npv = InputParameters()
ip_npv.setAsOfDate("2026-03-03")
ip_npv.setConventions(convs)            # section 2
ip_npv.setCurveConfigs(curve_configs)   # section 2
ip_npv.setTodaysMarketParams(tm)        # section 2
ip_npv.setPricingEngine(engine_data)    # section 2
ip_npv.setPortfolio(portfolio)          # section 5
ip_npv.setMarketConfigs(market_configs) # use xois config for pricing
ip_npv.setAllFixings(True)
ip_npv.setEntireMarket(True)
# setImplyTodaysFixings is omitted: historical overnight
# fixings (EUR-ESTER, USD-SOFR) are supplied explicitly in FIXINGS.
ip_npv.insertAnalytic("NPV")
ip_npv.setBaseCurrency("USD")
ip_npv.setResultsPath(str(OUTPUT_DIR))

md = StrVector(MARKET_DATA)
fx = StrVector(FIXINGS)

app_npv = OREApp(ip_npv, str(OUTPUT_DIR / "log_npv.txt"))
app_npv.run(MarketDataInMemoryLoader(ip_npv, md, fx))

print("Run time: %.2f sec" % app_npv.getRunTime())
print("Errors:  ", len(app_npv.getErrors()))

## 8 - Display NPV results

In [ ]:
def report_to_df(report):
    """Convert an ORE PlainInMemoryReport to a pandas DataFrame.
    The in-memory report is available via OREApp.getReport(name).
    """
    type_readers = {
        0: report.dataAsSize,
        1: report.dataAsReal,
        2: report.dataAsString,
        3: lambda i: [d.ISO() for d in report.dataAsDate(i)],
        4: report.dataAsPeriod,
    }
    return pd.DataFrame({
        report.header(i): list(type_readers[report.columnType(i)](i))
        for i in range(report.columns())
    })


In [ ]:
npv = report_to_df(app_npv.getReport("npv"))
cols = ["TradeId", "TradeType", "Maturity", "NPV", "NpvCurrency", "NPV(Base)", "BaseCurrency"]
cols = [c for c in cols if c in npv.columns]
npv[cols]

## 9 - Stress test: parallel +100 bp shift on USD and EUR discount curves

We build a single stress scenario (`parallel_rates_usd_eur`) that applies a
parallel +100 bp shift to both USD and EUR discount curves, then run it via
`InputParameters`, passing the Python-built scenario and simulation market
objects directly.

In [ ]:
TENORS = ["6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "15Y", "20Y"]
SHIFT  = 0.01  # 100 bp absolute

vr = DoubleVector([SHIFT] * len(TENORS))
vp = PeriodVector([Period(t) for t in TENORS])

shift_data = StressTestScenarioDataCurveShiftData()
shift_data.shiftType   = 1   # Absolute
shift_data.shifts      = vr
shift_data.shiftTenors = vp

stress_1 = StressTestScenarioDataStressTestData()
stress_1.setLabel("parallel_rates_usd_eur")
stress_1.setDiscountCurveShift("USD", shift_data)
stress_1.setDiscountCurveShift("EUR", shift_data)

stress_scenario_data = StressTestScenarioData()
stress_scenario_data.setData(stress_1)

# stress_xml = stress_scenario_data.toXMLString()
# print(stress_xml)

In [ ]:
# STRESS analytics requires a simulation market configuration.
# We build a minimal one in Python covering USD and EUR discount curves,
# index forwarding curves, and the EURUSD FX pair.
sim_params = ScenarioSimMarketParameters()
sim_params.setBaseCcy("USD")
sim_params.setDiscountCurveNames(StrVector(["USD", "EUR"]))
sim_params.setIndices(StrVector(["EUR-ESTER", "USD-SOFR"]))  # needed for xccy swap pricing
sim_params.setFxCcyPairs(StrVector(["EURUSD"]))

dc_tenors = PeriodVector([Period(t) for t in TENORS])
sim_params.setYieldCurveTenors("", dc_tenors)

sim_params.setSimulateFXVols(False)
sim_params.setSimulateSwapVols(False)
sim_params.setSimulateCapFloorVols(False)

sim_xml = sim_params.toXMLString()
# print(sim_xml[:1200])

In [ ]:
# Run stress using InputParameters with Python-built objects from sections 2, 5, 9.
ip_stress = InputParameters()
ip_stress.setAsOfDate("2026-03-03")
ip_stress.setConventions(convs)           # section 2
ip_stress.setCurveConfigs(curve_configs)   # section 2
ip_stress.setTodaysMarketParams(tm)        # section 2
ip_stress.setPricingEngine(engine_data)    # section 2
ip_stress.setPortfolio(portfolio)          # section 5
ip_stress.setMarketConfigs(market_configs) # use xois config for pricing
ip_stress.setAllFixings(True)
ip_stress.setEntireMarket(True)
ip_stress.setStressScenarioData(stress_scenario_data)    # section 9
ip_stress.setStressSimMarketParams(sim_params.toXMLString())  # section 9
ip_stress.insertAnalytic("STRESS")
ip_stress.setBaseCurrency("USD")
ip_stress.setResultsPath(str(OUTPUT_DIR))

md = StrVector(MARKET_DATA)
fx = StrVector(FIXINGS)

app_stress = OREApp(ip_stress, str(OUTPUT_DIR / "log_stress.txt"))
app_stress.run(MarketDataInMemoryLoader(ip_stress, md, fx))

print("Run time: %.2f sec" % app_stress.getRunTime())
print("Errors:  ", len(app_stress.getErrors()))

## 10 - Stress results

In [ ]:
stress_df = report_to_df(app_stress.getReport("stress"))
stress_df

## 11 - Summary

| ORE Python feature | Section |
|---|---|
| `ZeroRateConvention`, `FXConvention` in Python | 2 |
| `EngineData` pricing-engine configuration | 2 |
| `CurveConfigurations` + `DirectYieldCurveSegment` + `YieldCurveConfig` | 2, 4 |
| `TodaysMarketParameters` with `default` and `xois` configurations | 2 |
| `Conventions` inspection | 3 |
| `ScheduleData`, `FloatingLegData`, `LegData`, `ORESwap`: programmatic trade | 5 |
| Inline market data & fixings | 6 |
| `InputParameters` + `OREApp` + `MarketDataInMemoryLoader`: NPV | 7 |
| `StressTestScenarioData` object API | 9 |
| `ScenarioSimMarketParameters` object API | 9 |
| In-memory report extraction via `report_to_df` | 8, 10 |